In [ ]:
# importing libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from IPython.display import display

In [ ]:
# data directory relative to project root
data_dir = Path("../data/world_dev_index")

# loading data from the World Development Index dataset
## consumption data (final consumption expenditure (constant 2015 US$))
cdata = pd.read_excel(
    data_dir / "cdata.xls",
    skiprows=3,
)

## real interest rate data (real interest rate (%))
rdata = pd.read_excel(
    data_dir / "rdata.xls",
    skiprows=3,
)
## changing data-format from wide to long
cdata = cdata.melt(
    id_vars= ["Country Name", "Country Code"],
    value_vars = [c for c in cdata.columns if str(c).isdigit()],
    var_name = "year",
    value_name = "consumption"
)
rdata = rdata.melt(
    id_vars= ["Country Name", "Country Code"],
    value_vars= [r for r in rdata.columns if str(r).isdigit()],
    var_name="year",
    value_name="interest_rate"
)

# adjoining cdata and rdata
df = pd.merge(
    cdata,
    rdata,
    on=["Country Code", "Country Name", "year"]
)

# sorting by country and year
df = df.sort_values(["Country Code", "year"])
df

In [ ]:
df["c_growth"] = (
    df.groupby("Country Code")["consumption"].pct_change(periods=1, fill_method=None)
)

df["log_gross_c_growth"] = (
    np.log(1+ df["c_growth"])
)

df["r_lag1"] = (
    df.groupby("Country Code")["interest_rate"].shift(periods=1)
)

df["delta_r"] = (
    df.groupby("Country Code")["interest_rate"].diff(periods=1)
)

df["log_gross_r"] = (
    np.log(1+ df["interest_rate"])
)

df_cleaned = df.dropna()
df_cleaned


In [ ]:
model1_all_bm = sm.OLS.from_formula(
    "c_growth ~ interest_rate",
    data = df_cleaned
).fit()

model1_all_logr = sm.OLS.from_formula(
    "c_growth ~ log_r",
    data = df_cleaned
).fit()

model